In [2]:
# Import PyTorch and other relevant libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchsummary import summary

# MIT introduction to deep learning package
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

# other packages
import matplotlib.pyplot as plt
import numpy as np
import random
from tqdm import tqdm

!pip install comet_ml > /dev/null 2>&1
import comet_ml
# TODO: ENTER YOUR API KEY HERE!!
COMET_API_KEY = "fjFLr8fP0YKt0INjM2KlX2lkU"

# Check that we are using a GPU, if not switch runtimes
#   using Runtime > Change Runtime Type > GPU
assert torch.cuda.is_available(), "Please enable GPU from runtime settings"
assert COMET_API_KEY != "", "Please insert your Comet API Key"

# Set GPU for computation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/comet_ml/error_tracking/shutdown.py:22: SentryHubDeprecationWarning: `sentry_sdk.Hub` is deprecated and will be removed in a future major release. Please consult our 1.x to 2.x migration guide for details on how to migrate `Hub` usage to the new API: https://docs.sentry.io/pl

In [3]:
class Residual_Block(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, dropout=0.2):
        super(Residual_Block, self).__init__()
        
        padding = (kernel_size - 1) * dilation
        
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride, padding=0, dilation=dilation)
        self.pad =  nn.ConstantPad1d((padding, 0), 0.0)
        self.activation_function1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride, padding=0, dilation=dilation)
        self.activation_function2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.pad, self.conv1, self.activation_function1, self.dropout1, 
                                 self.pad, self.conv2 ,self.activation_function2, self.dropout2)
        
        if n_inputs != n_outputs:
            self.residual_connection = nn.Conv1d(n_inputs, n_outputs, 1)
        else:
            self.residual_connection = None

        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal(0, 0.01)
        self.conv2.weight.data.normal(0, 0.01)
        if self.residual_connection is not None:
            self.residual_connection.weight.data.normal(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        if self.residual_connection is None:
            residual = x
        else:
            residual = self.residual_connection(x)
        return nn.ReLU(out + residual)
        


        
        

In [1]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np


uploader = widgets.FileUpload(multiple=False)
display(uploader)



FileUpload(value={}, description='Upload')

In [2]:
import os
# get the uploaded file (first/only one)
name, meta = next(iter(uploader.value.items()))

out_path = f"/content/{name}"
with open(out_path, "wb") as f:
    f.write(meta["content"])

print("Saved to:", out_path)

Saved to: /content/08_undisturbed_fast_rotation_with_breaks_A_cleaned.hdf5


In [3]:
!pip install h5py

In [16]:
import h5py

with h5py.File("/content/08_undisturbed_fast_rotation_with_breaks_A_cleaned.hdf5", 'r') as f:
    
    d = f['imu_gyr']

    arr = np.asarray(d)

    mins = arr.min(axis=0)
    maxs = arr.max(axis=0)

    print(d)

    labels = ["x", "y", "z"]
    for i, lab in enumerate(labels):
        print(f"{lab}: min={mins[i]:.6f}, max={maxs[i]:.6f}")





<HDF5 dataset "imu_gyr": shape (55104, 3), type "<f8">
x: min=-23.599524, max=20.676348
y: min=-10.062795, max=9.352242
z: min=-8.305055, max=7.141751


In [ ]:
import csv
from itertools import islice

COLUMNS = [
    "Time",
    "attitude_roll",
    "attitude_pitch",
    "attitude_yaw",
    "rotation_rate_x",
    "rotation_rate_y",
    "rotation_rate_z",
    "gravity_x",
    "gravity_y",
    "gravity_z",
    "user_acc_x",
    "user_acc_y",
    "user_acc_z",
    "magnetic_field_x",
    "magnetic_field_y",
    "magnetic_field_z",
]

with open("/content/imu1.csv", newline="") as f:
    reader = csv.DictReader(f, fieldnames=COLUMNS)

    # "keys" (column names)
    print(reader.fieldnames)

    # example: print first 15 rotation_rate_y values (no magic numbers)
    for row in islice(reader, 15):
        print(float(row["gravity_y"]))


['Time', 'attitude_roll', 'attitude_pitch', 'attitude_yaw', 'rotation_rate_x', 'rotation_rate_y', 'rotation_rate_z', 'gravity_x', 'gravity_y', 'gravity_z', 'user_acc_x', 'user_acc_y', 'user_acc_z', 'magnetic_field_x', 'magnetic_field_y', 'magnetic_field_z']
-0.43429
-0.43455
-0.43499
-0.43544
-0.43574
-0.43579
-0.43564
-0.43538
-0.43513
-0.43488
-0.43456
-0.43421
-0.43384
-0.43345
-0.43312
